# 03 — NLP Models: Sentiment Analysis + Chatbot Training
Module B1 (text preprocessing), B2 (sentiment model), and B3 (chatbot intent classifier).
Trains on the sample data in `../data/` and saves artifacts to `../app/models/`.


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import json
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from app.services.nlp_service import clean_text

MODEL_DIR = "../app/models"
DATA_DIR = "../data"
os.makedirs(MODEL_DIR, exist_ok=True)


## B1 + B2. Text preprocessing & sentiment model

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/reviews_sample.csv")
print(df.head())

texts = [clean_text(t) for t in df["review"]]
labels = df["sentiment"].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

sentiment_model = LogisticRegression(max_iter=1000)
sentiment_model.fit(X_train_vec, y_train)

preds = sentiment_model.predict(X_test_vec)
print(classification_report(y_test, preds))

joblib.dump(sentiment_model, f"{MODEL_DIR}/sentiment_model.pkl")
joblib.dump(vectorizer, f"{MODEL_DIR}/vectorizer.pkl")
print("Saved sentiment_model.pkl + vectorizer.pkl")


## B3. Chatbot intent classifier (trained on `../data/intents.json`)

In [ ]:
with open(f"{DATA_DIR}/intents.json") as f:
    intents = json.load(f)

X_intent, y_intent = [], []
for tag, data in intents.items():
    for ex in data["examples"]:
        X_intent.append(clean_text(ex))
        y_intent.append(tag)

intent_vectorizer = TfidfVectorizer()
X_intent_vec = intent_vectorizer.fit_transform(X_intent)

intent_classifier = LogisticRegression(max_iter=1000)
intent_classifier.fit(X_intent_vec, y_intent)

joblib.dump(intent_classifier, f"{MODEL_DIR}/chatbot_model.pkl")
joblib.dump(intent_vectorizer, f"{MODEL_DIR}/chatbot_vectorizer.pkl")
print("Saved chatbot_model.pkl + chatbot_vectorizer.pkl")


In [ ]:
# Quick sanity checks
from app.services.nlp_service import analyze_sentiment
from app.services.chatbot_service import chatbot_reply

print(analyze_sentiment("This is the best purchase I've made all year!"))
print(chatbot_reply("Where's my package?"))
